In [ ]:
%%sql
CREATE OR REPLACE TEMP VIEW v_position_violations AS
WITH latest_pos_date AS (
    SELECT MAX(position_date) AS position_date
    FROM dbo.updated_positions
),
latest_limit_date AS (
    SELECT MAX(position_date) AS position_date
    FROM dbo.limit_positions_stddev
),
base_positions AS (
    SELECT
        p.position_date,
        p.trader_id,
        p.ticker,
        CAST(p.quantity AS DOUBLE) AS trader_qty
    FROM dbo.updated_positions p
    INNER JOIN latest_pos_date d
        ON p.position_date = d.position_date
),
base_limits AS (
    SELECT
        l.position_date,
        l.ticker,
        CAST(l.position_limit_qty AS DOUBLE) AS position_limit_qty
    FROM dbo.limit_positions_stddev l
    INNER JOIN latest_limit_date d
        ON l.position_date = d.position_date
)
SELECT
    p.position_date,
    p.trader_id,
    p.ticker,
    p.trader_qty,
    l.position_limit_qty,
    ABS(p.trader_qty) - l.position_limit_qty AS excess_qty
FROM base_positions p
INNER JOIN base_limits l
    ON p.ticker = l.ticker
WHERE ABS(p.trader_qty) > l.position_limit_qty;

In [ ]:
%%sql
CREATE OR REPLACE TABLE dbo.position_violations AS
SELECT *
FROM v_position_violations;

In [ ]:
%%sql
SELECT *
FROM dbo.position_violations
ORDER BY ticker, trader_id;